# Disk fraction of the W51 embedded population

Disk fraction estimated from the number ratio

$$R_{\rm disk} = \frac{N_{\rm YSO}}{N_{\rm lower}}, \qquad N_{\rm YSO} = N_{\rm upper} + N_{\rm very\,red},$$

where the YSO candidates are the upper-branch sources (infrared excess in F360M$-$F480M) plus the very red sources, and the lower-branch sources are the reddened population without excess. The conventional fraction $f_{\rm disk} = N_{\rm YSO}/(N_{\rm YSO} + N_{\rm lower})$ is listed next to it for reference.

Population definitions follow `color_magnitude_diagram_cleancat.ipynb` and `av_spatial_map_branches.ipynb`:

* $A_V$: F162M vs F162M$-$F210M CMD, CT06 MWLoc extinction vector slid back onto a straight-line fit to the 1 Myr PARSEC isochrone between 0.1 and 20 $M_\odot$ (`get_av`).
* **Foreground sources** ($A_V \leq 12$) are removed by the branch definitions themselves.
* **Upper branch**: $A_V > 12$ and $F360M-F480M > 0.15 + (1.5/6)(F162M-F210M)$. **Lower branch**: $A_V > 12$ and below that line.
* **Very red sources**: F162M or F210M missing and $F360M-F480M >$ `veryred_360m480_cut` (1 mag, as `missing_nir_idx_red` in the CMD notebook and `very_red_idx` in the NIRSpec MOS notebook). They have no CMD $A_V$, so they do not overlap with either branch.
* **Background sources**: sources where the ATLASGAL 870 $\mu$m continuum (`co_plank`, Jy/beam, on the F480M grid) is below `bg_870_threshold` = 1.5 Jy/beam, as in the CMD notebook.

Two spatial selections:

1. **Field without the background-dominated area**: every source inside the F480M footprint with $S_{870} \geq$ `bg_870_threshold`.
2. **Protocluster area**: $S_{870} \geq$ `protocluster_870_threshold`. With `protocluster_connected = True`, only the connected regions above the threshold that contain the W51-E or W51-IRS2 ALMA B3 field centers are kept, so isolated clumps elsewhere in the field are dropped.

Section 4 scans the 870 $\mu$m threshold, because the protocluster cut has no unique value.

In [ ]:
import os
FIGDIR = '/blue/adamginsburg/t.yoo/w51_nircam/analysis/plots_for_paper/figures/disk_fraction'
os.makedirs(FIGDIR, exist_ok=True)

from astropy.table import Table
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import astropy.units as u
from astroquery.svo_fps import SvoFps
from astropy.wcs import WCS
from astropy.io import fits
from astropy.stats import binom_conf_interval
from astropy.visualization import simple_norm
from dust_extinction.averages import CT06_MWLoc
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import ndimage

plt.rcParams['axes.labelsize'] = 20
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.linewidth'] = 3
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 3
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 3

jfilts = SvoFps.get_filter_list('JWST')
jfilts.add_index('filterID')


def get_mag(catalog, ww, filtername='f140m'):
    # Vega magnitudes from the catalog fluxes (same recipe as color_magnitude_diagram_cleancat.ipynb)
    flux = (catalog['flux_fit_' + filtername] * u.MJy/u.sr * ww.proj_plane_pixel_area()).to(u.Jy)
    eflux_jy = (catalog['flux_err_' + filtername] * u.MJy/u.sr * ww.proj_plane_pixel_area()).to(u.Jy)
    zeropoint_vega = u.Quantity(jfilts.loc[f'JWST/NIRCam.{filtername.upper()}']['ZeroPoint'], u.Jy)
    vegamag = -2.5 * np.log10(flux / zeropoint_vega) * u.mag
    vegamag_err = 2.5 / np.log(10) * np.abs(eflux_jy / flux) * u.mag
    return vegamag, vegamag_err


image_filenames = {
    "f162m": "/orange/adamginsburg/jwst/w51/images/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/images/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/images/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/images/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits",
    "f480m": "/orange/adamginsburg/jwst/w51/images/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
}
catalog = Table.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/catalogs/final_catalog_new_cfit_cut.fits')
catalog = catalog[catalog['nmatch_bands'] > 3]

headers = {filt: fits.getheader(fn, ext=('SCI', 1)) for filt, fn in image_filenames.items()}
f162 = get_mag(catalog, WCS(headers['f162m']), 'f162m')[0].to_value(u.mag)
f210 = get_mag(catalog, WCS(headers['f210m']), 'f210m')[0].to_value(u.mag)
f360 = get_mag(catalog, WCS(headers['f360m']), 'f360m')[0].to_value(u.mag)
f410 = get_mag(catalog, WCS(headers['f410m']), 'f410m')[0].to_value(u.mag)
f480 = get_mag(catalog, WCS(headers['f480m']), 'f480m')[0].to_value(u.mag)
print('N sources (nmatch_bands > 3):', len(catalog))

In [ ]:
# --- A_V from the F162M vs F162M-F210M CMD (same as av_estimate in the CMD notebook) ---
parsec_av0 = Table.read('/home/t.yoo/parsec_av0.txt', format='ascii')
parsec_av0 = parsec_av0[parsec_av0['col4'] < 300]
logage_av0 = parsec_av0['col3']
mass_av0 = parsec_av0['col6']
F162Mmag_av0 = parsec_av0['col40'] + 5*np.log10(5400) - 5
F210Mmag_av0 = parsec_av0['col42'] + 5*np.log10(5400) - 5

age_idx = logage_av0 == 6
idx_01msun = np.argmin(np.abs(mass_av0[age_idx] - 0.1))
idx_20msun = np.argmin(np.abs(mass_av0[age_idx] - 20))


def get_av(color, mag, ext=CT06_MWLoc(), color1=('f140m', 'f162m'), mag1=('f140m',), isochrone_line_params=None):
    m_iso, b_iso = isochrone_line_params
    w_color1_1 = int(color1[0][1:-1])/100*u.um
    w_color1_2 = int(color1[1][1:-1])/100*u.um
    w_mag1 = int(mag1[0][1:-1])/100*u.um
    e_color1_1 = ext(1/w_color1_1)
    e_color1_2 = ext(1/w_color1_2)
    e_mag1 = ext(1/w_mag1)

    # intersection of the extinction vector through (color, mag) with the isochrone line
    m_ext = e_mag1 / (e_color1_1 - e_color1_2)
    b_ext = mag - m_ext * color
    x_int = (b_ext - b_iso) / (m_iso - m_ext)
    y_int = m_iso * x_int + b_iso

    av = (mag - y_int) / e_mag1
    av[av < 0] = 0
    return av


isochrone_x_start = F162Mmag_av0[age_idx][idx_01msun] - F210Mmag_av0[age_idx][idx_01msun]
isochrone_x_end = F162Mmag_av0[age_idx][idx_20msun] - F210Mmag_av0[age_idx][idx_20msun]
isochrone_y_start = F162Mmag_av0[age_idx][idx_01msun]
isochrone_y_end = F162Mmag_av0[age_idx][idx_20msun]
color_slope = (isochrone_y_end - isochrone_y_start) / (isochrone_x_end - isochrone_x_start)
color_intercept = isochrone_y_start - color_slope * isochrone_x_start

av_estimate = get_av(f162 - f210, f162, ext=CT06_MWLoc(), color1=('f162m', 'f210m'), mag1=('f162m',),
                     isochrone_line_params=(color_slope, color_intercept))

In [ ]:
# --- populations: lower / upper branch (A_V > 12 removes the foreground) and very red sources ---
AV_MIN = 12
veryred_360m480_cut = 1.0  # mag, F360M-F480M cut of the very red sources (CMD / MOS notebooks)

with np.errstate(invalid='ignore'):
    branch_line = 0.15 + (1.5/6) * (f162 - f210)
    upper_mask = (f360 - f480 > branch_line) & (av_estimate > AV_MIN)
    lower_mask = (f360 - f480 <= branch_line) & (av_estimate > AV_MIN)
    missing_nir = ~np.isfinite(f162) | ~np.isfinite(f210)
    veryred_mask = missing_nir & (f360 - f480 > veryred_360m480_cut)
    # alternative very red cut used in av_spatial_map_branches.ipynb, for the comparison in Section 5
    veryred_mask_alt = missing_nir & (f410 - f480 > 0.5)
foreground_mask = np.isfinite(av_estimate) & (av_estimate <= AV_MIN)
yso_mask = upper_mask | veryred_mask

print(f'foreground (A_V <= {AV_MIN}): {foreground_mask.sum()}')
print(f'upper branch: {upper_mask.sum()}   lower branch: {lower_mask.sum()}   very red: {veryred_mask.sum()}')
print(f'overlap upper & very red: {(upper_mask & veryred_mask).sum()}   '
      f'overlap lower & very red: {(lower_mask & veryred_mask).sum()}')

In [ ]:
# --- F480M image, footprint, ATLASGAL 870 micron map, and S_870 at each source ---
wcs_f480m = WCS(headers['f480m'])
f480m_data = fits.getdata(image_filenames['f480m'], ext=('SCI', 1))
norm_f480m = simple_norm(f480m_data, stretch='log', percent=99.)
footprint = np.isfinite(f480m_data) & (f480m_data != 0)
pix_area_arcmin2 = wcs_f480m.proj_plane_pixel_area().to_value(u.arcmin**2)

# despite the variable name, this is the ATLASGAL 870um continuum map (Jy/beam), reprojected
# onto the F480M pixel grid -- see the note in color_magnitude_diagram_cleancat.ipynb
co_planck_fn = '/orange/adamginsburg/jwst/w51/data_reprojected_480/co_planck_reprojected_to_f480m.fits'
co_plank = fits.getdata(co_planck_fn, ext=0)
assert co_plank.shape == f480m_data.shape
# the ~19" ATLASGAL map is heavily oversampled on the F480M grid; contour every 8th pixel
CONTOUR_STEP = 8
co_plank_ds = co_plank[::CONTOUR_STEP, ::CONTOUR_STEP]
co_plank_x = np.arange(co_plank.shape[1])[::CONTOUR_STEP]
co_plank_y = np.arange(co_plank.shape[0])[::CONTOUR_STEP]

skycoord_ref = catalog['skycoord_ref']
pixcoord_ref = wcs_f480m.all_world2pix(skycoord_ref.ra.deg, skycoord_ref.dec.deg, 0)
ix_ref = np.round(pixcoord_ref[0])
iy_ref = np.round(pixcoord_ref[1])
on_map = (np.isfinite(ix_ref) & np.isfinite(iy_ref)
          & (ix_ref >= 0) & (ix_ref < co_plank.shape[1]) & (iy_ref >= 0) & (iy_ref < co_plank.shape[0]))
ix_ref = np.where(on_map, ix_ref, 0).astype(int)
iy_ref = np.where(on_map, iy_ref, 0).astype(int)
on_map &= footprint[iy_ref, ix_ref]
s870_at_source = np.where(on_map, co_plank[iy_ref, ix_ref], np.nan)

print(f'sources outside the F480M footprint (dropped from both selections): {(~on_map).sum()}')
for name, m in [('upper', upper_mask), ('lower', lower_mask), ('very red', veryred_mask)]:
    print(f'  {name:9s} outside: {(m & ~on_map).sum()}')
print(f'F480M footprint area: {footprint.sum() * pix_area_arcmin2:.2f} arcmin^2')


def format_wcs_axes(ax):
    # tick formatting only -- axis labels are set per figure
    ra = ax.coords['ra']
    dec = ax.coords['dec']
    ra.set_major_formatter('hh:mm:ss.ss')
    dec.set_major_formatter('dd:mm:ss.ss')
    ra.set_ticklabel(rotation=90, pad=30)
    ra.set_ticks_position('l')
    ra.set_ticklabel_position('l')
    ra.set_axislabel('')
    dec.set_ticks_position('b')
    dec.set_ticklabel_position('b')
    dec.set_axislabel('')

In [ ]:
# --- W51-E / W51-IRS2 positions: center of each ALMA B3 long-baseline continuum image ---
# --- (same images as in color_magnitude_diagram_cleancat.ipynb and the NIRSpec MOS notebook) ---
alma_b3_dir = '/orange/adamginsburg/w51/TaehwaYoo/2017.1.00293.S_W51_B3_LB/may2021_successful_imaging/'
alma_cont_filenames = {
    'W51-E': alma_b3_dir + 'w51e2.spw0thru19.14500.robust0.thr0.075mJy.mfs.I.startmod.selfcal7.image.tt0.pbcor.fits',
    'W51-IRS2': alma_b3_dir + 'w51n.spw0thru19.14500.robust0.thr0.075mJy.mfs.I.startmod.selfcal7.image.tt0.pbcor.fits',
}


def alma_image_center(fn):
    h = fits.getheader(fn)
    return WCS(h).celestial.pixel_to_world((h['NAXIS1'] - 1) / 2, (h['NAXIS2'] - 1) / 2)


protocluster_pix = {}
for name, fn in alma_cont_filenames.items():
    px, py = (float(v) for v in wcs_f480m.world_to_pixel(alma_image_center(fn)))
    protocluster_pix[name] = (px, py)
    print(f'{name:9s} pixel ({px:.0f}, {py:.0f})   S_870 = {co_plank[int(round(py)), int(round(px))]:.1f} Jy/beam')

## 1. Spatial selections

* `bg_870_threshold`: background-dominated area boundary (1.5 Jy/beam, CMD notebook).
* `protocluster_870_threshold`: protocluster boundary. The default of 10 Jy/beam is a working choice; Section 4 shows how the result depends on it.
* `protocluster_connected`: keep only the connected regions above `protocluster_870_threshold` that contain W51-E or W51-IRS2.

In [ ]:
bg_870_threshold = 1.5            # Jy/beam
protocluster_870_threshold = 10.  # Jy/beam
protocluster_connected = True


def region_from_threshold(threshold, connected=protocluster_connected):
    # pixel mask of S_870 >= threshold inside the F480M footprint; optionally only the
    # connected regions that contain one of the protocluster centers
    region = footprint & (co_plank >= threshold)
    if not connected:
        return region
    labels, _ = ndimage.label(region)
    keep = {labels[int(round(py)), int(round(px))] for px, py in protocluster_pix.values()} - {0}
    return np.isin(labels, list(keep))


def sources_in(region):
    return on_map & region[iy_ref, ix_ref]


region_field = footprint & (co_plank >= bg_870_threshold)
region_protocluster = region_from_threshold(protocluster_870_threshold)
selections = {
    'field (870um >= bg)': (region_field, f'$S_{{870}} \\geq$ {bg_870_threshold} Jy/beam (excl. background area)'),
    'protocluster': (region_protocluster, f'protocluster: $S_{{870}} \\geq$ {protocluster_870_threshold:g} Jy/beam'
                     + (' (connected to W51-E / IRS2)' if protocluster_connected else '')),
}
for key, (region, label) in selections.items():
    print(f'{key:22s} area = {region.sum() * pix_area_arcmin2:6.2f} arcmin^2')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 15), subplot_kw={'projection': wcs_f480m})
pops = [('lower branch', lower_mask, 'blue'), ('upper branch', upper_mask, 'orange'),
        ('very red sources', veryred_mask, 'r')]
for axi, (key, (region, label)) in zip(axes, selections.items()):
    axi.imshow(f480m_data, origin='lower', cmap='gray', norm=norm_f480m)
    region_ds = region[::CONTOUR_STEP, ::CONTOUR_STEP].astype(float)
    axi.contourf(co_plank_x, co_plank_y, region_ds, levels=[0.5, 1.5], colors='cyan', alpha=0.15)
    axi.contour(co_plank_x, co_plank_y, region_ds, levels=[0.5], colors='cyan', linewidths=1.5)
    sel = sources_in(region)
    for name, m, color in pops:
        axi.scatter(pixcoord_ref[0][m & sel], pixcoord_ref[1][m & sel], s=4, c=color, alpha=0.7,
                    label=f'{name} (N={(m & sel).sum()})')
    for name, (px, py) in protocluster_pix.items():
        axi.scatter(px, py, s=250, marker='+', color='lime', linewidths=2.5, zorder=6)
        axi.annotate(name, (px, py), color='lime', fontsize=14, fontweight='bold',
                     xytext=(10, 10), textcoords='offset points')
    handles, _ = axi.get_legend_handles_labels()
    handles.append(Patch(facecolor='cyan', alpha=0.3, edgecolor='cyan', label='selected area'))
    axi.legend(handles=handles, markerscale=4, fontsize=12, loc='upper right', framealpha=0.8)
    axi.set_xlim(-0.5, f480m_data.shape[1] - 0.5)
    axi.set_ylim(-0.5, f480m_data.shape[0] - 0.5)
    format_wcs_axes(axi)
    axi.set_xlabel('Declination', fontsize=16)
    axi.set_ylabel('Right Ascension', fontsize=16)
    axi.set_title(label, fontsize=16)
fig.subplots_adjust(hspace=0.25)
plt.savefig(f'{FIGDIR}/disk_fraction_selection_maps_cfit003.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Disk fraction in the two selections

Uncertainties:

* $R_{\rm disk} = N_{\rm YSO}/N_{\rm lower}$: Poisson errors propagated, $\sigma_R = R\sqrt{1/N_{\rm YSO} + 1/N_{\rm lower}}$.
* $f_{\rm disk} = N_{\rm YSO}/(N_{\rm YSO}+N_{\rm lower})$: 68.3% Wilson binomial interval.

These are counting errors only. Classification errors (branch-line placement, the very red color cut, $A_V$ errors near 12 mag) and incompleteness are not included; Section 5 shows the effect of the very red cut.

In [ ]:
def disk_fraction(sel, yso=None, lower=None):
    yso = yso_mask if yso is None else yso
    lower = lower_mask if lower is None else lower
    n_up, n_vr = (upper_mask & sel).sum(), ((yso & ~upper_mask) & sel).sum()
    n_yso, n_low = (yso & sel).sum(), (lower & sel).sum()
    ratio = n_yso / n_low
    ratio_err = ratio * np.sqrt(1 / n_yso + 1 / n_low)
    frac = n_yso / (n_yso + n_low)
    frac_lo, frac_hi = binom_conf_interval(n_yso, n_yso + n_low, confidence_level=0.683, interval='wilson')
    return dict(N_upper=n_up, N_veryred=n_vr, N_YSO=n_yso, N_lower=n_low,
                R_disk=ratio, R_disk_err=ratio_err, f_disk=frac, f_disk_lo=frac_lo, f_disk_hi=frac_hi)


rows = []
for key, (region, label) in selections.items():
    r = disk_fraction(sources_in(region))
    rows.append(dict(selection=key, area_arcmin2=region.sum() * pix_area_arcmin2, **r))
# for reference: the full footprint, background area included (foreground still removed)
rows.append(dict(selection='full footprint (incl. bg)', area_arcmin2=footprint.sum() * pix_area_arcmin2,
                 **disk_fraction(on_map)))
result = Table(rows)
for col in ['area_arcmin2', 'R_disk', 'R_disk_err', 'f_disk', 'f_disk_lo', 'f_disk_hi']:
    result[col].format = '.3f'
result.pprint(max_width=-1)

for r in result:
    print(f"{r['selection']:26s} R_disk = {r['R_disk']:.2f} +/- {r['R_disk_err']:.2f}   "
          f"f_disk = {r['f_disk']:.3f} (+{r['f_disk_hi'] - r['f_disk']:.3f} / -{r['f_disk'] - r['f_disk_lo']:.3f})")
result.write(f'{FIGDIR}/disk_fraction_summary_cfit003.ecsv', overwrite=True)

## 3. Background area: what is removed

The populations inside the background-dominated area ($S_{870} <$ `bg_870_threshold`). The lower branch loses a much larger share there than the YSO candidates, so including this area lowers $R_{\rm disk}$ (compare the "full footprint" row above).

In [ ]:
bg_sel = on_map & (s870_at_source < bg_870_threshold)
for name, m in [('upper branch', upper_mask), ('very red', veryred_mask), ('lower branch', lower_mask)]:
    print(f'{name:13s} in background area: {(m & bg_sel).sum():5d} of {(m & on_map).sum():5d} '
          f'({(m & bg_sel).sum() / (m & on_map).sum():.1%})')
r_bg = disk_fraction(bg_sel)
print(f"background area alone: R_disk = {r_bg['R_disk']:.2f} +/- {r_bg['R_disk_err']:.2f}")

## 4. Dependence on the 870 $\mu$m threshold

$R_{\rm disk}$ and $f_{\rm disk}$ for a range of 870 $\mu$m thresholds, with all sources above the threshold (whole footprint) and with only the connected regions containing W51-E / W51-IRS2. The shaded band is the counting error. Vertical lines mark `bg_870_threshold` and `protocluster_870_threshold`.

In [ ]:
thresholds = np.array([1.5, 2, 3, 4, 5, 7.5, 10, 12.5, 15, 20, 25, 30, 40])
scan = {}
for connected in [False, True]:
    out = []
    for t in thresholds:
        region = region_from_threshold(t, connected=connected)
        r = disk_fraction(sources_in(region))
        out.append(dict(threshold=t, area_arcmin2=region.sum() * pix_area_arcmin2, **r))
    scan[connected] = Table(out)
    print('connected to W51-E / IRS2' if connected else 'whole footprint above threshold')
    scan[connected]['threshold', 'area_arcmin2', 'N_upper', 'N_veryred', 'N_lower', 'R_disk', 'R_disk_err',
                    'f_disk'].pprint(max_width=-1)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for connected, color, label in [(False, 'k', 'whole footprint'), (True, 'm', 'connected to W51-E / IRS2')]:
    t = scan[connected]
    axes[0].plot(t['threshold'], t['R_disk'], 'o-', color=color, lw=2, label=label)
    axes[0].fill_between(t['threshold'], t['R_disk'] - t['R_disk_err'], t['R_disk'] + t['R_disk_err'],
                         color=color, alpha=0.2, lw=0)
    axes[1].plot(t['threshold'], t['f_disk'], 'o-', color=color, lw=2, label=label)
    axes[1].fill_between(t['threshold'], t['f_disk_lo'], t['f_disk_hi'], color=color, alpha=0.2, lw=0)
for ax, ylab in zip(axes, [r'$R_{\rm disk} = N_{\rm YSO}/N_{\rm lower}$',
                           r'$f_{\rm disk} = N_{\rm YSO}/(N_{\rm YSO}+N_{\rm lower})$']):
    ax.axvline(bg_870_threshold, color='gray', ls='--', lw=1.5)
    ax.axvline(protocluster_870_threshold, color='gray', ls=':', lw=1.5)
    ax.set_xscale('log')
    ax.set_xlabel(r'$S_{870}$ threshold (Jy/beam)')
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.4)
axes[0].legend(fontsize=14, frameon=False)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/disk_fraction_vs_870um_threshold_cfit003.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Sensitivity to the very red source definition

The very red sources make up a large part of $N_{\rm YSO}$, and the two population notebooks use different cuts (F360M$-$F480M $> 1$ here and in the CMD notebook; F410M$-$F480M $> 0.5$ in `av_spatial_map_branches.ipynb`). This cell recomputes the two selections with each cut and with the upper branch alone (no very red sources).

In [ ]:
variants = [
    (f'upper + very red (F360M-F480M > {veryred_360m480_cut:g})', upper_mask | veryred_mask),
    ('upper + very red (F410M-F480M > 0.5)', upper_mask | veryred_mask_alt),
    ('upper branch only', upper_mask),
]
for key, (region, label) in selections.items():
    sel = sources_in(region)
    print(key)
    for name, yso in variants:
        r = disk_fraction(sel, yso=yso)
        print(f"  {name:42s} N_YSO = {r['N_YSO']:5d}  N_lower = {r['N_lower']:5d}  "
              f"R_disk = {r['R_disk']:.2f} +/- {r['R_disk_err']:.2f}  f_disk = {r['f_disk']:.3f}")